In [ ]:
import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Seed fixed to 42 — results will be reproducible")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from PIL import Image
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
!apt-get install unrar -qq

source_path = '/content/drive/MyDrive/fair disease'
extract_path = '/content/data/extracted'
os.makedirs(extract_path, exist_ok=True)

for rar_file in sorted(os.listdir(source_path)):
    if rar_file.endswith('.rar'):
        full_path = os.path.join(source_path, rar_file)
        folder_name = rar_file.replace('.rar', '')
        dest = os.path.join(extract_path, folder_name)
        os.makedirs(dest, exist_ok=True)
        !unrar x -o+ "{full_path}" "{dest}/"
        print(f"Extracted: {rar_file}")

print("\nExtraction complete!")

In [ ]:
class LungCTDataset(Dataset):
    def __init__(self, data_dirs, target_depth=64, target_size=256, is_train=True):
        self.samples = []
        self.target_depth = target_depth
        self.target_size = target_size
        self.is_train = is_train

        for folder_path, label in data_dirs:
            for gender in ["male", "female"]:
                gender_path = os.path.join(folder_path, gender)
                if not os.path.exists(gender_path):
                    continue
                gender_id = 1 if gender == "male" else 0
                for scan_name in os.listdir(gender_path):
                    if scan_name.startswith("._"):
                        continue
                    scan_path = os.path.join(gender_path, scan_name)
                    if os.path.isdir(scan_path):
                        self.samples.append((scan_path, label, gender_id))

        males = sum(1 for _, _, g in self.samples if g == 1)
        females = sum(1 for _, _, g in self.samples if g == 0)
        print(f"Loaded {len(self.samples)} samples (Male: {males}, Female: {females})")

    def _load_scan(self, scan_path):
        slice_files = []
        for f in os.listdir(scan_path):
            if f.startswith("._") or not f.endswith(".jpg"):
                continue
            slice_num = int(f.replace(".jpg", ""))
            slice_files.append((slice_num, f))
        slice_files.sort(key=lambda x: x[0])

        slices = []
        for _, fname in slice_files:
            img = Image.open(os.path.join(scan_path, fname)).convert("L")
            img = img.resize((self.target_size, self.target_size))
            slices.append(np.array(img, dtype=np.float32))

        if len(slices) == 0:
            return np.zeros((1, self.target_size, self.target_size), dtype=np.float32)
        return np.stack(slices, axis=0)

    def _remove_non_lung_slices(self, volume):
        D = volume.shape[0]
        if D <= 5:
            return volume
        start = int(D * 0.10)
        end = int(D * 0.90)
        if end <= start:
            return volume
        return volume[start:end]

    def _resize_depth(self, volume, target_depth):
        D = volume.shape[0]
        if D == 0:
            return np.zeros((target_depth, volume.shape[1], volume.shape[2]), dtype=np.float32)
        indices = np.linspace(0, D - 1, target_depth).astype(int)
        return volume[indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        scan_path, label, gender = self.samples[idx]

        volume = self._load_scan(scan_path)
        volume = self._remove_non_lung_slices(volume)
        volume = self._resize_depth(volume, self.target_depth)
        volume = volume / 255.0

        if self.is_train:
            crop_size = 224
            h_start = np.random.randint(0, self.target_size - crop_size + 1)
            w_start = np.random.randint(0, self.target_size - crop_size + 1)
            volume = volume[:, h_start:h_start+crop_size, w_start:w_start+crop_size]

            if np.random.random() > 0.5:
                k = np.random.choice([1, 2, 3])
                volume = np.rot90(volume, k, axes=(1, 2)).copy()

            if np.random.random() > 0.5:
                factor = np.random.uniform(0.8, 1.2)
                volume = np.clip(volume * factor, 0, 1)
        else:
            crop_size = 224
            h_start = (self.target_size - crop_size) // 2
            w_start = (self.target_size - crop_size) // 2
            volume = volume[:, h_start:h_start+crop_size, w_start:w_start+crop_size]

        volume = torch.FloatTensor(volume).unsqueeze(0)
        return volume, label, gender


data_root = "/content/data/extracted"

train_dirs = [
    (os.path.join(data_root, "train1", "A"), 0),
    (os.path.join(data_root, "train1", "G"), 1),
    (os.path.join(data_root, "train2", "covid"), 2),
    (os.path.join(data_root, "train2", "normal"), 3),
]

val_dirs = [
    (os.path.join(data_root, "validation for fair diagonosis", "val", "A"), 0),
    (os.path.join(data_root, "validation for fair diagonosis", "val", "G"), 1),
    (os.path.join(data_root, "validation for fair diagonosis", "val", "covid"), 2),
    (os.path.join(data_root, "validation for fair diagonosis", "val", "normal"), 3),
]

train_dataset = LungCTDataset(train_dirs, target_depth=64, target_size=256, is_train=True)
val_dataset = LungCTDataset(val_dirs, target_depth=64, target_size=256, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

sample_vol, sample_label, sample_gender = train_dataset[0]
print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Sample shape: {sample_vol.shape}, label: {sample_label}, gender: {'male' if sample_gender==1 else 'female'}")

In [ ]:
from torchvision.models.video import r3d_18, R3D_18_Weights

class LungDiagnosisModel3D(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        base_model = r3d_18(weights=R3D_18_Weights.DEFAULT)

        old_conv = base_model.stem[0]
        new_conv = nn.Conv3d(1, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2),
                            padding=(1, 3, 3), bias=False)
        with torch.no_grad():
            new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        base_model.stem[0] = new_conv

        self.features = nn.Sequential(
            base_model.stem,
            base_model.layer1,
            base_model.layer2,
            base_model.layer3,
            base_model.layer4,
        )
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

model = LungDiagnosisModel3D(num_classes=4).to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class FairLACVaRLoss(nn.Module):
    def __init__(self, class_counts, alpha=0.5, tau=1.0):
        super().__init__()
        counts = torch.tensor(class_counts, dtype=torch.float32)
        freqs = counts / counts.sum()
        log_prior = tau * torch.log(freqs)
        self.register_buffer('log_prior', log_prior)
        self.alpha = alpha

    def la_cross_entropy(self, logits, targets):
        adjusted_logits = logits + self.log_prior
        return F.cross_entropy(adjusted_logits, targets, reduction='none')

    def _binary_search_lambda(self, group_losses, num_iters=32):
        lo = group_losses.min().item()
        hi = group_losses.max().item()
        for _ in range(num_iters):
            mid = (lo + hi) / 2.0
            frac_above = (group_losses > mid).float().mean().item()
            deriv = 1.0 - frac_above / self.alpha
            if deriv < 0:
                lo = mid
            else:
                hi = mid
        return (lo + hi) / 2.0

    def forward(self, logits, targets, group_labels):
        per_sample_loss = self.la_cross_entropy(logits, targets)

        unique_groups = torch.unique(group_labels)
        group_losses = []
        for g in unique_groups:
            mask = (group_labels == g)
            if mask.sum() > 0:
                group_losses.append(per_sample_loss[mask].mean())

        if len(group_losses) == 1:
            return group_losses[0]

        group_losses = torch.stack(group_losses)
        num_groups = group_losses.shape[0]

        if self.alpha >= 1.0:
            return group_losses.mean()

        lam = self._binary_search_lambda(group_losses)
        hinge = F.relu(group_losses - lam)
        loss = lam + hinge.sum() / (self.alpha * num_groups)
        return loss


class_counts = [0, 0, 0, 0]
for _, label, _ in train_dataset.samples:
    class_counts[label] += 1

class_names = ["Adenocarcinoma", "Squamous Cell", "COVID-19", "Normal"]
print("Class Distribution:")
for name, count in zip(class_names, class_counts):
    print(f"  {name}: {count}")

total = sum(class_counts)
freqs = [c / total for c in class_counts]
print(f"\nClass frequencies: {[f'{f:.4f}' for f in freqs]}")
print(f"Log-prior offsets: {[f'{np.log(f):.4f}' for f in freqs]}")

criterion = FairLACVaRLoss(
    class_counts=class_counts,
    alpha=0.7,
    tau=1.0
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

print(f"\nFair LA+CVaR Loss ready (alpha={criterion.alpha}, tau=1.0)")

In [ ]:
num_epochs = 80
scaler = GradScaler()

train_losses = []
train_accs = []

os.makedirs("/content/checkpoints", exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for volumes, labels, genders in train_loader:
        volumes = volumes.to(device)
        labels = labels.to(device)
        genders = genders.to(device)

        optimizer.zero_grad()
        with autocast():
            outputs = model(volumes)
            loss = criterion(outputs, labels, genders)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    scheduler.step()
    train_acc = 100.0 * train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)

    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)

    torch.save(model.state_dict(), f"/content/checkpoints/model_epoch_{epoch+1}.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2f}% | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nTraining complete! All {num_epochs} checkpoints saved.")

In [ ]:
best_f1 = 0.0
best_epoch = 0
val_f1s = []

for epoch in range(1, num_epochs + 1):
    model.load_state_dict(torch.load(f"/content/checkpoints/model_epoch_{epoch}.pth"))
    model.eval()

    epoch_preds = []
    epoch_labels = []

    with torch.no_grad():
        for volumes, labels, _genders in val_loader:
            volumes = volumes.to(device)
            labels = labels.to(device)
            with autocast():
                outputs = model(volumes)
            _, predicted = outputs.max(1)
            epoch_preds.extend(predicted.cpu().numpy())
            epoch_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(epoch_labels, epoch_preds, average='macro')
    val_f1s.append(macro_f1)
    print(f"Epoch {epoch} | Val Macro F1: {macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_epoch = epoch

model.load_state_dict(torch.load(f"/content/checkpoints/model_epoch_{best_epoch}.pth"))
torch.save(model.state_dict(), "/content/best_model_fair.pth")
print(f"\nBest Epoch: {best_epoch} | Best Macro F1: {best_f1:.4f}")
print("Best model saved as best_model_fair.pth")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(range(1, len(train_losses)+1), train_losses, 'b-', linewidth=2)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(train_accs)+1), train_accs, 'g-', linewidth=2)
axes[1].set_title('Training Accuracy', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(range(1, len(val_f1s)+1), val_f1s, 'r-', linewidth=2)
axes[2].set_title('Validation Macro F1 Score', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro F1')
axes[2].axvline(x=best_epoch, color='black', linestyle='--', label=f'Best Epoch ({best_epoch})')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_plots_fair.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plots saved!")

In [ ]:
model.load_state_dict(torch.load("/content/best_model_fair.pth"))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for volumes, labels, _genders in val_loader:
        volumes = volumes.to(device)
        labels = labels.to(device)
        with autocast():
            outputs = model(volumes)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)
all_labels_np = np.array(all_labels)
all_preds_np = np.array(all_preds)

class_names = ["Adenocarcinoma", "Squamous Cell", "COVID-19", "Normal"]

print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=class_names))

print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(all_labels, all_preds)
for i, name in enumerate(class_names):
    print(f"{name:20s}: {cm[i]}")

macro_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"\nFinal Macro F1 Score: {macro_f1:.4f}")

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

cm = confusion_matrix(all_labels, all_preds)
im = axes[0].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set_title('Confusion Matrix', fontsize=14)
plt.colorbar(im, ax=axes[0])
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].set_xticklabels(class_names, rotation=45, ha='right')
axes[0].set_yticklabels(class_names)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

for i in range(4):
    for j in range(4):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=14)

labels_bin = label_binarize(all_labels_np, classes=[0, 1, 2, 3])
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

for i, (name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_title('ROC Curves (One-vs-Rest)', fontsize=14)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/evaluation_plots_fair.png", dpi=150, bbox_inches='tight')
plt.show()
print("Evaluation plots saved!")

In [ ]:
model.load_state_dict(torch.load("/content/best_model_fair.pth"))
model.eval()

male_preds, male_labels = [], []
female_preds, female_labels = [], []

with torch.no_grad():
    for volumes, labels, genders in val_loader:
        volumes = volumes.to(device)
        with autocast():
            outputs = model(volumes)
        preds = outputs.argmax(dim=1).cpu().numpy()
        labels_np = labels.numpy()
        genders_np = genders.numpy()

        for p, l, g in zip(preds, labels_np, genders_np):
            if g == 1:
                male_preds.append(p)
                male_labels.append(l)
            else:
                female_preds.append(p)
                female_labels.append(l)

f1_male = f1_score(male_labels, male_preds, average='macro')
f1_female = f1_score(female_labels, female_preds, average='macro')
final_score = 0.5 * (f1_male + f1_female)

class_names = ["Adenocarcinoma", "Squamous Cell", "COVID-19", "Normal"]

print("=" * 60)
print("GENDER-WISE EVALUATION (FAIRNESS)")
print("=" * 60)
print(f"\nMale samples:   {len(male_labels)}")
print(f"Female samples: {len(female_labels)}")
print(f"\nMacro F1 (Male):   {f1_male:.4f}")
print(f"Macro F1 (Female): {f1_female:.4f}")
print(f"F1 Gap:            {abs(f1_male - f1_female):.4f}")
print(f"\nFinal Score = 1/2 (F1_male + F1_female) = {final_score:.4f}")
print("=" * 60)

print("\n--- Male Classification Report ---")
print(classification_report(male_labels, male_preds, target_names=class_names))

print("--- Female Classification Report ---")
print(classification_report(female_labels, female_preds, target_names=class_names))

In [ ]:
import shutil
from datetime import datetime

alpha_val = str(criterion.alpha).replace(".", "")
save_dir = f"/content/drive/MyDrive/fair disease/submission_fair_alpha{alpha_val}"
os.makedirs(save_dir, exist_ok=True)

shutil.copy("/content/best_model_fair.pth", os.path.join(save_dir, "best_model_fair.pth"))
shutil.copy("/content/training_plots_fair.png", os.path.join(save_dir, "training_plots_fair.png"))
shutil.copy("/content/evaluation_plots_fair.png", os.path.join(save_dir, "evaluation_plots_fair.png"))

with open(os.path.join(save_dir, "results_fair.txt"), "w") as f:
    f.write("=" * 70 + "\n")
    f.write("  Fair Lung Disease Diagnosis — LA + CVaR Loss\n")
    f.write("  Fair Disease Diagnosis Challenge\n")
    f.write("=" * 70 + "\n\n")

    f.write("METHODOLOGY\n")
    f.write("-" * 70 + "\n")
    f.write("Architecture  : 3D ResNet-18 with Pretrained Kinetics-400 Weights\n")
    f.write("Loss Function : Logit-Adjusted CE + CVaR Fairness Aggregation\n")
    f.write(f"  tau   = 1.0\n")
    f.write(f"  alpha = {criterion.alpha}\n")
    f.write("Optimizer     : Adam (lr=1e-4, weight_decay=1e-5)\n")
    f.write("Scheduler     : CosineAnnealing (T_max=30)\n")
    f.write(f"Date          : {datetime.now().strftime('%B %d, %Y')}\n\n")

    f.write("RESULTS\n")
    f.write("-" * 70 + "\n")
    f.write(f"Best Epoch         : {best_epoch}\n")
    f.write(f"Overall Macro F1   : {macro_f1:.4f}\n")
    f.write(f"Macro F1 (Male)    : {f1_male:.4f}\n")
    f.write(f"Macro F1 (Female)  : {f1_female:.4f}\n")
    f.write(f"F1 Gap             : {abs(f1_male - f1_female):.4f}\n")
    f.write(f"Fair Score (avg)   : {final_score:.4f}\n\n")

    f.write("CLASSIFICATION REPORT\n")
    f.write("-" * 70 + "\n")
    f.write(classification_report(all_labels, all_preds, target_names=class_names))

print(f"All results saved to: {save_dir}")

In [ ]:
f1_gap = abs(f1_male - f1_female)

print("=" * 60)
print("PERFORMANCE ASSESSMENT (Competition Format)")
print("=" * 60)
print(f"\nSubset A (Male)   — Macro F1 : {f1_male:.4f}")
print(f"Subset B (Female) — Macro F1 : {f1_female:.4f}")
print(f"\nFinal Score = 1/2 (F1_male + F1_female)")
print(f"            = 1/2 ({f1_male:.4f} + {f1_female:.4f})")
print(f"            = {final_score:.4f}")
print(f"\nF1 Gap (|male - female|) = {f1_gap:.4f}")
print("=" * 60)

alpha_val = str(criterion.alpha).replace(".", "")
save_dir = f"/content/drive/MyDrive/fair disease/submission_fair_alpha{alpha_val}"
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, "final_score.txt"), "w") as f:
    f.write("=" * 60 + "\n")
    f.write("PERFORMANCE ASSESSMENT — Fair Disease Diagnosis Challenge\n")
    f.write("=" * 60 + "\n\n")
    f.write("Method: Logit-Adjusted CE + CVaR Fairness Loss\n")
    f.write(f"  tau   = 1.0\n")
    f.write(f"  alpha = {criterion.alpha}\n")
    f.write(f"  Best Epoch = {best_epoch}\n\n")
    f.write("-" * 60 + "\n")
    f.write(f"Subset A (Male)   — Macro F1 : {f1_male:.4f}\n")
    f.write(f"Subset B (Female) — Macro F1 : {f1_female:.4f}\n\n")
    f.write(f"Final Score = 1/2 (F1_male + F1_female)\n")
    f.write(f"            = 1/2 ({f1_male:.4f} + {f1_female:.4f})\n")
    f.write(f"            = {final_score:.4f}\n\n")
    f.write(f"F1 Gap (|male - female|) = {f1_gap:.4f}\n")
    f.write("-" * 60 + "\n\n")
    f.write("Per-Gender Detailed Reports:\n\n")
    f.write("--- Male ---\n")
    f.write(classification_report(male_labels, male_preds, target_names=class_names))
    f.write("\n--- Female ---\n")
    f.write(classification_report(female_labels, female_preds, target_names=class_names))

print(f"\nSaved to: {save_dir}/final_score.txt")